# Week 30, Day 4: The cost of a spike: overlapping currents and temperature

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w030/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Decomposing the spike

`spike(phi)` simulates a spike evoked by a 0.3 ms pulse of 100 μA/cm² and returns the voltage and the sodium and potassium currents, with all rates multiplied by phi. At phi = 1, compute the inward sodium charge `Q_Na` (nC/cm², the integral of the negative part of INa), the minimum charge `Q_min` = C (V_peak - V_rest) with C = 1 μF/cm², and their `ratio`.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
dt = 0.002

def rates_f(V, phi=1.0):
    V = V + 1e-7
    r = (0.01 * (V + 55) / (1 - math.exp(-(V + 55) / 10)), 0.125 * math.exp(-(V + 65) / 80),
         0.1 * (V + 40) / (1 - math.exp(-(V + 40) / 10)), 4.0 * math.exp(-(V + 65) / 18),
         0.07 * math.exp(-(V + 65) / 20), 1 / (1 + math.exp(-(V + 35) / 10)))
    return [phi * x for x in r]

def spike(phi, T=10.0):
    V, n, m, h = -65.0, 0.3177, 0.0529, 0.5961
    Vs, INa, IK = [], [], []
    for k in range(int(T / dt)):
        an, bn, am, bm, ah, bh = rates_f(V, phi)
        ina, ik = 120 * m**3 * h * (V - 50), 36 * n**4 * (V + 77)
        I = 100.0 if 1 <= k * dt < 1.3 else 0.0
        n += dt * (an * (1 - n) - bn * n); m += dt * (am * (1 - m) - bm * m); h += dt * (ah * (1 - h) - bh * h)
        V += dt * (I - ina - ik - 0.3 * (V + 54.387))
        Vs.append(V); INa.append(ina); IK.append(ik)
    return np.array(Vs), np.array(INa), np.array(IK)

V, INa, IK = spike(1.0)
Q_Na = ...
Q_min = ...
ratio = Q_Na / Q_min
print(f"peak {V.max():.1f} mV; sodium charge {Q_Na:.0f} nC/cm^2, minimum {Q_min:.0f} nC/cm^2, ratio {ratio:.1f}")
tt = np.arange(len(V)) * dt
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(tt, V); ax[0].set_ylabel("V (mV)")
ax[1].plot(tt, INa, label="I_Na"); ax[1].plot(tt, IK, label="I_K"); ax[1].set_ylabel("uA/cm^2"); ax[1].legend()
ax[1].set_xlabel("ms"); plt.show()

In [ ]:
import numpy as np
def test_charges():
    "Q_Na is the inward sodium charge and Q_min is C times the spike's amplitude from rest"
    assert abs(Q_Na - (-np.sum(np.minimum(INa, 0)) * dt)) < 1e-9 and abs(Q_min - (max(V) + 65)) < 1e-9

def test_ratio():
    "at 6.3 C the sodium entry is more than ten times the minimum"
    assert ratio > 10

test_charges()
test_ratio()
print("All checks passed")

### Exercise: Warming the squid

For temperatures 6.3, 10, 18.5 and 26 °C, compute phi, run `spike(phi)`, and store in `table[T]` the tuple (phi, peak voltage, half-width in ms, sodium ratio). The half-width is the time the voltage spends above the midpoint between rest (-65 mV) and the peak.

In [ ]:
table = {}
for T in (6.3, 10, 18.5, 26):
    phi = ...
    V, INa, IK = spike(phi)
    peak = V.max()
    half_width = ...
    ratio_T = (-np.sum(np.minimum(INa, 0)) * dt) / (peak + 65)
    table[T] = (phi, peak, half_width, ratio_T)
    print(f"{T:4.1f} C: phi {phi:5.2f}, peak {peak:5.1f} mV, half-width {half_width:.2f} ms, sodium {ratio_T:.1f} x minimum")

In [ ]:
import numpy as np
def test_phi():
    "phi is 3 raised to (T - 6.3) / 10"
    assert all(abs(table[T][0] - 3 ** ((T - 6.3) / 10)) < 1e-9 for T in table)

def test_trends():
    "the ratio falls from above 10 at 6.3 C to between 3 and 5 at 18.5 C, and the spike narrows"
    assert table[6.3][3] > 10 and 3 < table[18.5][3] < 5 and table[26][2] < table[6.3][2] / 3

test_phi()
test_trends()
print("All checks passed")

### Exercise: What a spike costs

A cortical pyramidal neuron has about 20,000 μm² (2 × 10⁻⁴ cm²) of membrane that each spike charges by 100 mV (a simplification: in reality the spike spreads into only part of the dendrites, but the axon adds area). Write `atp(f)` giving ATP molecules per spike for an overlap factor f, and compare f = 1.3 (mammalian axons) with f = 13 (the squid model at 6.3 °C).

In [ ]:
e, c_m, dV, area = 1.602e-19, 1e-6, 0.1, 2e-4      # C, F/cm^2, V, cm^2

def atp(f):
    return ...

for f in (1.3, 4, 13):
    print(f"overlap {f:4.1f}: {atp(f):.1e} ATP per spike, {atp(f) * 8e-20 * 1e12:.1f} pJ")

In [ ]:
import numpy as np
def test_atp():
    "atp gives the overlap factor times C dV times the area, divided by e and by 3"
    assert abs(atp(1.3) - 1.3 * 1e-6 * 0.1 * 2e-4 / 1.602e-19 / 3) < 1

def test_compare():
    "the mammalian-like spike costs about 5e7 ATP and the squid-like one about ten times more"
    assert 3e7 < atp(1.3) < 8e7 and 8 < atp(13) / atp(1.3) < 12

test_atp()
test_compare()
print("All checks passed")